# 02 — Multi-hop Retrieval
Mini-hop seed 문서 본문과 Claim에서 실제 Wikipedia 제목을 찾고, HoVer Train으로 fine-tune한 문맥 RoBERTa가 경로를 재정렬합니다. 2/3/4-Hop은 동일 beam-search 코드가 목표 Hop까지 반복 확장합니다. Evidence와 Dev Gold는 후보 생성에 사용하지 않습니다.

In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ['PYTHONUNBUFFERED']='1'
import base64,io,zipfile
BUNDLE_B64='...'
if Path('/content').is_dir():
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as archive: archive.extractall('/content')
    os.environ['HOVER_E2E_ROOT']='/content/e2e_v2'
    print('[bootstrap] latest e2e_v2 code synchronized to /content/e2e_v2',flush=True)
DRIVE_CANDIDATE=Path('/content/drive/MyDrive/hover-project')
DRIVE_PROJECT=DRIVE_CANDIDATE if DRIVE_CANDIDATE.is_dir() else None
roots=([Path(os.environ['HOVER_E2E_ROOT']).expanduser()] if os.environ.get('HOVER_E2E_ROOT') else [])+[Path.cwd(),*Path.cwd().parents,Path('/content/e2e_v2'),Path('/content/고준서/e2e_v2'),Path('/content/Fact-Verfication')]+([DRIVE_PROJECT] if DRIVE_PROJECT else [])
candidates=[candidate for root in roots for candidate in (root,root/'e2e_v2',root/'고준서/e2e_v2')]
E2E_ROOT=next((p.resolve() for p in candidates if (p/'pyproject.toml').is_file() and (p/'src/hover_e2e').is_dir()),None)
if E2E_ROOT is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_PROJECT=Path('/content/drive/MyDrive/hover-project')
        mounted_candidates=[DRIVE_PROJECT,DRIVE_PROJECT/'e2e_v2',DRIVE_PROJECT/'고준서/e2e_v2']
        E2E_ROOT=next((p.resolve() for p in mounted_candidates if (p/'pyproject.toml').is_file() and (p/'src/hover_e2e').is_dir()),None)
    except ImportError:
        pass
if E2E_ROOT is None: raise FileNotFoundError('현재 커널과 마운트된 Drive에서 e2e_v2 소스를 찾지 못했습니다. /content/e2e_v2 또는 MyDrive/hover-project/고준서/e2e_v2에 폴더가 필요합니다.')
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{E2E_ROOT}[gpu,test]'])
sys.path.insert(0,str(E2E_ROOT/'src')) if str(E2E_ROOT/'src') not in sys.path else None
from hover_e2e.colab_runtime import prepare_runtime,backup_shared_artifacts
RUN_NAME=os.environ.get('HOVER_RUN_NAME','roberta_e2e_v2')
HOPS=os.environ.get('HOVER_HOPS','2')
CTX=prepare_runtime(E2E_ROOT,drive_project=DRIVE_PROJECT,run_name=RUN_NAME)
CONFIG=E2E_ROOT/'configs/base.json'
import torch
assert torch.cuda.is_available(),'GPU 런타임으로 변경하세요.'
print(torch.cuda.get_device_name(0),'| hops',HOPS)


In [ ]:
subprocess.run([sys.executable,'-u','-m','hover_e2e.cli','multihop','--config',str(CONFIG),'--data-root',str(CTX.data_root),'--output-root',str(CTX.output_root),'--run-name',RUN_NAME,'--hops',HOPS],check=True)
backup_shared_artifacts(CTX)
print('Multi-hop complete. 다음: 03_evidence.ipynb')
